# Data Explorer: DABStep Benchmark Inference

This notebook demonstrates the **leaderboard-winning** Data Explorer agent on the [DABStep benchmark](https://huggingface.co/datasets/adyen/DABstep) (Data Agent Benchmark for Multi-step Reasoning).

**Results: #1 on the DABStep Leaderboard**

| System | Easy | Hard | Time/Task | Code Length |
|--------|------|------|-----------|-------------|
| **Data Explorer + Nemotron 3 Super (Ours)** | **87.5** | **~77** | **20s** | **1,870** |
| Data Explorer + Haiku 4.5 (Ours) | 87.5 | 89.95 | 20s | 1,870 |
| Claude Code + Opus 4.5 | 90.2 | 66.93 | 10 min | 5,011 |
| DataPilot (AntGroup) | 86.11 | 87.57 | -- | -- |
| DS-STAR (Google AI) | 87.5 | 45.24 | -- | -- |

Our approach achieves **30x speedup** over the baseline while **dominating the Hard category** (84% of the benchmark).

## The Three-Phase Architecture

```
Phase 1: LEARNING                Phase 2: INFERENCE              Phase 3: REFLECTION
┌─────────────────┐             ┌─────────────────┐             ┌─────────────────┐
│ Heavy Model     │             │ Light Model     │             │ Heavy Model     │
│ (Opus 4.5/4.6)  │             │ (Nemotron Super)│             │ (Opus/Sonnet)   │
│                 │             │                 │             │                 │
│ Multi-pass loop │   distill   │ Single-pass loop│   review    │ Reflection +    │
│ + ground truth  │───────────▶ │ + helper.py     │───────────▶ │ Self-consistency│
│ + full toolkit  │             │ + few-shot only │             │ (offline)       │
└─────────────────┘             └─────────────────┘             └─────────────────┘
     ▼ output:                       ▼ output:                       ▼ output:
  helper.py                       answer + trace                  improved prompts
  solutions.md                                                    fed back to Phase 2
```

**Key insight**: By investing upfront in learning and code abstraction (Phase 1), even a small, fast model can outsmart heavier models on complex multi-step problems (Phase 2).

## Setup

The DABStep inference server should already be running on port 8000. Let's verify the connection and check the data.

In [ ]:
import os
import json
import requests
import time
from dotenv import load_dotenv

os.chdir('/app')
load_dotenv()

SERVER_URL = 'http://localhost:8000'

assert os.environ.get('NV_INFER_API_KEY'), (
    'NV_INFER_API_KEY not set. Create a .env file with your key or export it: '
    'export NV_INFER_API_KEY=your_key_here'
)

# Wait for server to be ready (it starts in the background)
for attempt in range(10):
    try:
        resp = requests.get(f'{SERVER_URL}/docs', timeout=5)
        if resp.status_code == 200:
            print(f'DABStep server is ready at {SERVER_URL}')
            break
    except requests.ConnectionError:
        if attempt < 9:
            print(f'Waiting for server... (attempt {attempt + 1}/10)')
            time.sleep(5)
        else:
            print('Server not responding. Start it manually with:')
            print('  uv run python dabstep_agent/inference/server.py &')

## Explore the DABStep Dataset

DABStep is a financial payments dataset with 450 tasks focused on merchant fees, transactions, and domain-specific rules. Let's peek at the data.

In [ ]:
import pandas as pd

payments = pd.read_csv('/app/data/context/payments.csv')
print(f'Payments: {len(payments):,} transactions')
print(f'Columns: {", ".join(payments.columns)}')
print()
payments.head(3)

In [ ]:
with open('/app/data/context/fees.json') as f:
    fees = json.load(f)
print(f'Fee rules: {len(fees)}')
print(f'Sample rule keys: {", ".join(fees[0].keys())}')
print()
print('First fee rule:')
json.dumps(fees[0], indent=2)

In [ ]:
# Load sample questions from the dev set
tasks_path = '/app/data/tasks_dev.json'
if os.path.exists(tasks_path):
    with open(tasks_path) as f:
        tasks = json.load(f)
    print(f'Dev tasks: {len(tasks)}')
    print()
    for i, t in enumerate(tasks[:5]):
        print(f'Task {t.get("task_id", i)}: {t["question"][:100]}...')
        print(f'  Answer: {t.get("answer", "N/A")}')
        print()
else:
    print(f'Dev tasks not found at {tasks_path}.')
    print('Run: uv run python dabstep_agent/download_data.py')

## The Distilled Knowledge

During the **Learning phase**, a heavyweight model solved training tasks and distilled the solutions into:
- `helper.py` -- a library of reusable fee-calculation functions
- `new_solutions.md` -- few-shot examples showing how to use the helper

This is what makes inference fast: the agent only needs function signatures, not the underlying code.

In [ ]:
# Show the helper.py function signatures
import inspect
import importlib.util

spec = importlib.util.spec_from_file_location('helper', '/app/dabstep_agent/inference/helper.py')
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)

public_funcs = [name for name, obj in inspect.getmembers(helper)
                if inspect.isfunction(obj) and not name.startswith('_')]

print(f'helper.py provides {len(public_funcs)} functions:\n')
for name in public_funcs:
    sig = inspect.signature(getattr(helper, name))
    doc = (getattr(helper, name).__doc__ or '').strip().split('\n')[0]
    print(f'  {name}{sig}')
    if doc:
        print(f'    {doc}')
    print()

## Ask the Agent a Question

Let's send a question to the running DABStep inference server. The agent (Nemotron 3 Super) uses the distilled `helper.py` and few-shot examples to solve tasks in seconds.

In [ ]:
def ask_dabstep(question: str, guidelines: str = 'N/A') -> dict:
    """Send a question to the DABStep inference server."""
    t0 = time.time()
    resp = requests.post(
        f'{SERVER_URL}/solve',
        json={'question': question, 'guidelines': guidelines},
        timeout=300
    )
    elapsed = time.time() - t0
    result = resp.json()
    result['elapsed_seconds'] = round(elapsed, 1)
    return result


# Example: a straightforward question
result = ask_dabstep(
    question='Which issuing country has the highest number of transactions?'
)

print(f'Answer: {result["agent_answer"]}')
print(f'Time:   {result["elapsed_seconds"]}s')
print(f'Trace:  {len(result["reasoning_trace"])} messages')

In [ ]:
# View the reasoning trace (what the agent did step by step)
for msg in result['reasoning_trace']:
    role = msg.get('role', '?')
    content = msg.get('content', '')
    tool = msg.get('tool_name', '')

    if role == 'human':
        print(f'[PROMPT] {content[:200]}...')
    elif role == 'ai':
        if msg.get('tool_calls'):
            for tc in msg['tool_calls']:
                code = tc.get('args', {}).get('code', '')[:200]
                print(f'[AGENT] Tool call: {tc.get("name", "?")}  →  {code}...')
        elif content:
            print(f'[AGENT] {content[:300]}')
    elif role == 'tool':
        print(f'[TOOL:{tool}] {content[:200]}')
    print()

## More Examples

Try some harder questions that require multi-step reasoning, fee calculations, and cross-referencing multiple data sources.

In [ ]:
hard_questions = [
    ('What is the average transaction value grouped by shopper_interaction for '
     "Crossfit_Hanna's TransactPlus transactions between January and April 2023?",
     "['POS: 89.34', 'Ecommerce: 92.70']"),
    ('What is the fee ID or IDs that apply to account_type = H and aci = D?',
     '5, 9, 20, 30, 31, 41, 46, 48, ... (161 IDs)'),
    ('For the 200th of the year 2023, what is the total fees (in euros) that Rafa_AI should pay?',
     '52.37'),
]

for q, expected in hard_questions:
    print(f'Q: {q}')
    r = ask_dabstep(q)
    print(f'A: {r["agent_answer"]}  ({r["elapsed_seconds"]}s)')
    print(f'Expected: {expected}')
    print('-' * 60)

## Before vs After: Brute Force vs Learned Pipeline

On the DABStep leaderboard, the baseline approach (Claude Code + Opus 4.5) solves every task **from scratch** -- no reusable code, no domain knowledge. It takes **10 minutes per task** and scores **66.93% on hard questions**.

Our Data Explorer approach invests upfront in a **Learning phase** that distills domain logic into reusable `helper.py` functions and few-shot examples. At inference time, even a lightweight open-weight model (Nemotron 3 Super) solves tasks in seconds with high accuracy -- a **30x speedup** without sacrificing quality.

Below we demonstrate this live. We run 3 hard multi-step questions two ways:
1. **From scratch** -- the agent has no helper functions, no examples, and must figure out the domain logic on its own
2. **With learned pipeline** -- the agent uses the distilled `helper.py` and few-shot examples built during the Learning phase

In [ ]:
import asyncio
from nat.runtime.loader import load_workflow
from data_explorer_agent.python_executor import _tools as executor_tools

NAIVE_CONFIG = '/app/dabstep_agent/inference/dabstep_config.yml'

comparison_questions = [
    {
        'question': 'For the 12th of the year 2023, what is the total fees (in euros) that Belles_cookbook_store should pay?',
        'answer': '12.91',
    },
    {
        'question': 'What were the applicable Fee IDs for Rafa_AI in December 2023?',
        'answer': '36, 51, 65, 107, 123, 141, 150, 163, 183, 276, 286, 304, 359, 384, 427, 428, 454, 477, 498, 595, 626, 631, 634, 701, 709, 741, 787, 813, 861, 888, 892, 924',
    },
    {
        'question': 'Looking at the month of July, to which card scheme should the merchant Martinis_Fine_Steakhouse steer traffic in order to pay the minimum fees?',
        'answer': 'NexPay:347.81',
    },
]

async def ask_naive(question: str) -> dict:
    """Run a question through the agent WITHOUT helper.py or few-shot examples."""
    naive_prompt = f"""Available data files in 'data/context/':
- payments.csv (CSV): 138K transactions with columns like merchant, card_scheme, eur_amount, is_credit, has_fraudulent_dispute, ip_country, issuing_country, etc.
- fees.json (JSON): 1000 fee rules with fields like ID, card_scheme, account_type, fixed_amount, rate, is_credit, aci, etc.
- merchant_data.json (JSON): 30 merchants with fields like merchant, account_type, merchant_category_code, capture_delay
- manual.md: Domain documentation about fee rules, fraud, and payment processing

No helper functions are available. Write all code from scratch.

QUESTION: {question}
GUIDELINES: N/A"""

    t0 = time.time()
    try:
        async with load_workflow(NAIVE_CONFIG) as wf:
            async with wf.run(naive_prompt) as runner:
                try:
                    raw = await asyncio.wait_for(runner.result(), timeout=120)
                except asyncio.TimeoutError:
                    raw = "TIMEOUT"
    except Exception as e:
        raw = f"FAILED: {type(e).__name__}"
    elapsed = time.time() - t0

    try:
        await executor_tools['reset_environment']()
    except:
        pass

    import re
    answer = str(raw).strip()
    m = re.search(r'"agent_answer"\s*:\s*"([^"]*)"', answer)
    if m:
        answer = m.group(1)

    return {'answer': answer[:200], 'time': round(elapsed, 1)}


print('Running 3 HARD questions WITHOUT distilled knowledge...')
print('(no helper.py, no few-shot examples -- agent must figure it out from scratch)\n')

naive_results = []
for item in comparison_questions:
    q = item['question']
    print(f'Q: {q[:90]}...')
    r = await ask_naive(q)
    naive_results.append(r)
    print(f'A: {r["answer"][:80]}  ({r["time"]}s)')
    print(f'Expected: {item["answer"]}')
    print('-' * 60)

In [ ]:
print('Now running the SAME 3 questions WITH distilled knowledge...\n')

learned_results = []
for item in comparison_questions:
    q = item['question']
    print(f'Q: {q[:90]}...')
    r = ask_dabstep(q)
    learned_results.append(r)
    print(f'A: {r["agent_answer"][:80]}  ({r["elapsed_seconds"]}s)')
    print('-' * 60)

In [ ]:
print('=' * 100)
print('COMPARISON: From-Scratch Brute Force  vs  Learned Pipeline (Data Explorer)')
print('=' * 100)
print()

total_naive_t = 0
total_learned_t = 0

for i, item in enumerate(comparison_questions):
    q = item['question']
    expected = item['answer']
    naive_a = naive_results[i]['answer']
    naive_t = naive_results[i]['time']
    learned_a = learned_results[i]['agent_answer']
    learned_t = learned_results[i]['elapsed_seconds']
    total_naive_t += naive_t
    total_learned_t += learned_t

    print(f'Question {i+1}: {q}')
    print()
    print(f'  Correct Answer:             {expected}')
    print(f'  From Scratch (no learning): {naive_a:<50}  [{naive_t}s]')
    print(f'  Learned Pipeline:           {learned_a:<50}  [{learned_t}s]')
    print()
    print('-' * 100)
    print()

print()
print(f'Total time from scratch:   {total_naive_t:.1f}s')
print(f'Total time with learning:  {total_learned_t:.1f}s')
if total_learned_t > 0:
    print(f'Speedup:                   {total_naive_t / total_learned_t:.1f}x')
else:
    print(f'Speedup:                   N/A')
print()
print('At benchmark scale (450 tasks), this translates to:')
print(f'  From-scratch baseline (Claude Code + Opus 4.5): ~10 min/task → 75 hours total')
print(f'  Data Explorer learned pipeline (Haiku 4.5):     ~20 sec/task → 2.5 hours total')
print(f'  Hard accuracy: 66.93% (baseline) vs 89.95% (Data Explorer)')
print()
print('The Learning phase pays for itself many times over at inference scale.')

## Try Your Own Questions

The DABStep dataset covers financial payments: merchants, fees, transactions, card schemes, and fraud. Ask anything about the data below.

In [ ]:
# Type your own question here:
# result = ask_dabstep('Your question about the DABStep financial dataset here')
# print(f'Answer: {result["agent_answer"]}')